## Divisão da planilha 'dataset_long.csv' em 7 novas planilhas

In [1]:
import pandas as pd

from pathlib import Path

# ---- Configuração ----
INPUT_FILE = Path('../data/dataset_pre_processado.csv')
# Os sete CSVs gerados ficam diretamente em data/, na raiz do projeto.
OUTPUT_DIR = Path('../data')
OUTPUT_DIR.mkdir(exist_ok=True)

# Colunas removidas de todos os arquivos de saída, além de 'key_area'
# (decisão do grupo — ver Card H)
COLUMNS_TO_DROP = ['run_id', 'tc_number', 'reason', 'active_limit', 'key_area']

# Termopares excluídos ANTES de qualquer tratamento (decisão do grupo), por área:
# - Oven RTD Connector Body (Mainback): erro de coleta do parceiro
#   (ver discussão de outliers / seção 4.2.2 da documentação).
# - AC Blower Inlet Max e AC Blower Inlet Avg (Main Vent Sys): medidos em só 4 dos
#   56 ensaios (7,1 %) e com tc_number inválido (#REF!) na base. Sem a exclusão, a
#   imputação por constante preencheria 93 % da coluna e o termopar passaria no
#   corte de cobertura dos modelos como se fosse uma medição real
#   (ver relatorio_dados_nao_treinados.md).
EXCLUDED_TC_BY_AREA = {
    'Mainback': ['Oven RTD Connector Body'],
    'Main Vent Sys': ['AC Blower Inlet Max', 'AC Blower Inlet Avg'],
}

# Cobertura mínima de um termopar para virar alvo dos modelos: o mesmo corte dos
# notebooks de treino e de scripts/export_models.py (MIN_TARGET_COVERAGE).
COBERTURA_MINIMA_ALVO = 0.50

# Grafias divergentes da base: o valor da esquerda é unificado ao da direita ANTES
# de dividir por key_area, para que os modelos já recebam as categorias
# padronizadas e não precisem corrigi-las em cada notebook.
# blower_direction ("To back"/"To front") e losses ("No losses") ficam de fora de
# propósito: a base já traz essas categorias na grafia correta, e trocá-las as
# afastaria dela.
GRAFIAS_DIVERGENTES = {
    "run_case":         {"Broil Case 2": "Broil C2", "Broil Qual": "Broil Qualitative", "Qualitative": "Broil Qualitative"},
    "housing":          {"Open - Updated": "Open"},
    # O glossario oficial do parceiro so documenta 2 niveis para Material Coil
    # Support (Aluminium / Plastic). 1 run (48) ja vinha rotulado "Aluminium
    # Plate" no dado bruto, antes de qualquer reclassificacao nossa - sem
    # confirmacao fisica desse nivel extra, normaliza de volta para Aluminium
    # (ver documents/extras/hipotese_material_sem_correcao_E17.md).
    "material_coil_support": {"Aluminium Plate": "Aluminium "},
}


In [2]:
def split_by_key_area(df: pd.DataFrame, output_dir: Path) -> tuple[dict, dict]:

    summary = {}
    area_dataframes = {}
    key_areas = df['key_area'].unique()
    print(f'Áreas encontradas ({len(key_areas)}): {list(key_areas)}\n')

    for area in key_areas:
        filename = output_dir / f"{area.lower().replace(' ', '_')}_data.csv"
        area_df = df[df['key_area'] == area].copy()

        excluded_tcs = EXCLUDED_TC_BY_AREA.get(area, [])
        if excluded_tcs and 'tc_name' in area_df.columns:
            area_df = area_df[~area_df['tc_name'].isin(excluded_tcs)]

        area_df = area_df.drop(columns=[c for c in COLUMNS_TO_DROP if c in area_df.columns])

        if 'temp' in area_df.columns:
            ordered_cols = [c for c in area_df.columns if c != 'temp'] + ['temp']
            area_df = area_df[ordered_cols]

        area_df.to_csv(filename, index=False)
        area_dataframes[area] = area_df
        summary[area] = len(area_df)
        print(f'"{filename.name}" salvo com {len(area_df)} linhas.')

    return summary, area_dataframes

In [3]:
df = pd.read_csv(INPUT_FILE)

# Unifica as grafias divergentes (GRAFIAS_DIVERGENTES) antes da divisão por área.
registro_grafias = []
for variavel, correcoes in GRAFIAS_DIVERGENTES.items():
    for origem, destino in correcoes.items():
        registro_grafias.append({
            'Variável': variavel,
            'Grafia original': origem,
            'Grafia padronizada': destino,
            'Linhas alteradas': int(df[variavel].eq(origem).sum()),
        })
    df[variavel] = df[variavel].replace(correcoes)
display(pd.DataFrame(registro_grafias))

print(f"Dataset principal carregado: {len(df)} linhas, {df['key_area'].nunique()} key_areas.\n")

summary, key_area_dataframes = split_by_key_area(df, OUTPUT_DIR)

# DataFrames finais, um para cada key_area.
mainback_df = key_area_dataframes['Mainback']
coil_temps_df = key_area_dataframes['Coil Temps']
main_vent_sys_df = key_area_dataframes['Main Vent Sys']
ipc_cooling_df = key_area_dataframes['IPC Cooling']
ipc_comp_df = key_area_dataframes['IPC Comp']
oven_hmi_df = key_area_dataframes['Oven HMI']
ct_knobs_df = key_area_dataframes['CT Knobs']

,Variável,Grafia original,Grafia padronizada,Linhas alteradas
0,run_case,Broil Case 2,Broil C2,0
1,run_case,Broil Qual,Broil Qualitative,0
2,run_case,Qualitative,Broil Qualitative,0
3,housing,Open - Updated,Open,0
4,material_coil_support,Aluminium Plate,Aluminium,74


Dataset principal carregado: 4101 linhas, 7 key_areas.

Áreas encontradas (7): ['Mainback', 'Coil Temps', 'Main Vent Sys', 'IPC Cooling', 'IPC Comp', 'Oven HMI', 'CT Knobs']

"mainback_data.csv" salvo com 557 linhas.
"coil_temps_data.csv" salvo com 420 linhas.
"main_vent_sys_data.csv" salvo com 280 linhas.
"ipc_cooling_data.csv" salvo com 218 linhas.
"ipc_comp_data.csv" salvo com 1979 linhas.
"oven_hmi_data.csv" salvo com 220 linhas.
"ct_knobs_data.csv" salvo com 366 linhas.


In [4]:
# ---- Verificação de integridade ----
total_saved = sum(summary.values())
excluded_rows = 0
if 'tc_name' in df.columns:
    for area, excluded_tcs in EXCLUDED_TC_BY_AREA.items():
        for tc_name in excluded_tcs:
            n_excluded = int((df['key_area'].eq(area) & df['tc_name'].eq(tc_name)).sum())
            excluded_rows += n_excluded
            print(f'Excluído de {area}: {tc_name} ({n_excluded} linhas)')
expected = len(df) - excluded_rows

print('Resumo por área:')
for area, n in summary.items():
    print(f'  {area}: {n} linhas')

print(f"\nTotal salvo: {total_saved} | Esperado (total original - excluídas): {expected}")
assert total_saved == expected, 'Divergência na contagem de linhas! Verifique o processamento.'
print('Contagem de linhas conferida com sucesso — nenhuma linha perdida ou duplicada além do esperado.')

Excluído de Mainback: Oven RTD Connector Body (53 linhas)
Excluído de Main Vent Sys: AC Blower Inlet Max (4 linhas)
Excluído de Main Vent Sys: AC Blower Inlet Avg (4 linhas)
Resumo por área:
  Mainback: 557 linhas
  Coil Temps: 420 linhas
  Main Vent Sys: 280 linhas
  IPC Cooling: 218 linhas
  IPC Comp: 1979 linhas
  Oven HMI: 220 linhas
  CT Knobs: 366 linhas

Total salvo: 4040 | Esperado (total original - excluídas): 4040
Contagem de linhas conferida com sucesso — nenhuma linha perdida ou duplicada além do esperado.


In [5]:
df.head(100)

,run_id,tc_number,tc_name,key_area,active_limit,temp,reason,blower_direction,housing,board_position,run_case,losses,deflector_hole,material_coil_support
0,Run 1,4,"Door Lock Motor Ambient 1/2"" Below",Mainback,155,99.35,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
1,Run 1,5,Door Lock Microswitch Body,Mainback,136,114.72,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
2,Run 1,6,Convect Motor Winding,Mainback,165,89.93,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
3,Run 1,#REF!,Storage Drawer (Metal Above),Mainback,150,167.87,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
4,Run 1,#REF!,"L2, Wire, Power Cord",Mainback,60,53.65,Não informado,To back,Open,Center,Pyro,No losses,Open,Aluminium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,Run 2 - Pyro Flipped,#REF!,IGBT 3,IPC Comp,110,92.10,Não informado,To front,Open,Center,Pyro,No losses,Open,Aluminium
96,Run 2 - Pyro Flipped,#REF!,IGBT 4,IPC Comp,110,92.06,Não informado,To front,Open,Center,Pyro,No losses,Open,Aluminium
97,Run 2 - Pyro Flipped,#REF!,IGBT 5,IPC Comp,110,91.99,Não informado,To front,Open,Center,Pyro,No losses,Open,Aluminium
98,Run 2 - Pyro Flipped,#REF!,IGBT 6,IPC Comp,110,91.96,Não informado,To front,Open,Center,Pyro,No losses,Open,Aluminium


## Converter 'tc_name' em colunas

In [6]:
def tc_names_to_columns(area_df: pd.DataFrame) -> pd.DataFrame:
    """Transforma tc_name em colunas, preservando todas as temperaturas."""
    index_columns = [column for column in area_df.columns if column not in ['tc_name', 'temp']]
    pivot_source = area_df.copy()

    # Numera medições repetidas para não descartar temperaturas no pivot.
    pivot_source['measurement_number'] = (
        pivot_source.groupby(index_columns + ['tc_name'], dropna=False).cumcount()
    )

    wide_df = (
        pivot_source.pivot(
            index=index_columns + ['measurement_number'],
            columns='tc_name',
            values='temp',
        )
        .reset_index()
    )
    wide_df.columns.name = None

    # A numeração é apenas auxiliar para o pivot e não faz parte da planilha final.
    wide_df = wide_df.drop(columns='measurement_number')


    return wide_df

key_area_wide_dataframes = {
    area: tc_names_to_columns(area_df)
    for area, area_df in key_area_dataframes.items()
}

mainback_wide_df = key_area_wide_dataframes['Mainback']
coil_temps_wide_df = key_area_wide_dataframes['Coil Temps']
main_vent_sys_wide_df = key_area_wide_dataframes['Main Vent Sys']
ipc_cooling_wide_df = key_area_wide_dataframes['IPC Cooling']
ipc_comp_wide_df = key_area_wide_dataframes['IPC Comp']
oven_hmi_wide_df = key_area_wide_dataframes['Oven HMI']
ct_knobs_wide_df = key_area_wide_dataframes['CT Knobs']

for area, wide_df in key_area_wide_dataframes.items():
    filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
    wide_df.to_csv(filename, index=False)
    print(f'"{filename.name}" salvo com {len(wide_df)} linhas e {len(wide_df.columns)} colunas.')

"mainback_data.csv" salvo com 56 linhas e 17 colunas.
"coil_temps_data.csv" salvo com 56 linhas e 15 colunas.
"main_vent_sys_data.csv" salvo com 56 linhas e 12 colunas.
"ipc_cooling_data.csv" salvo com 55 linhas e 11 colunas.
"ipc_comp_data.csv" salvo com 56 linhas e 44 colunas.
"oven_hmi_data.csv" salvo com 55 linhas e 11 colunas.
"ct_knobs_data.csv" salvo com 73 linhas e 14 colunas.


## Tratamento de dados nulos

Será utilizado a mediana caso a coluna do componente contenha dados nulos, caso contrário será utilizado a média.

In [7]:
def fill_component_nulls(wide_df: pd.DataFrame, component_columns) -> dict:
    """Fill temperature nulls using median for IQR outliers, otherwise mean."""
    # Guarda qual método foi usado em cada coluna de componente.
    treatment_by_component = {}

    # Analisa cada componente (cada tc_name virou uma coluna).
    for component in component_columns:
        # Converte valores para numéricos; valores inválidos passam a ser nulos.
        temperatures = pd.to_numeric(wide_df[component], errors='coerce')
        valid_temperatures = temperatures.dropna()
        nulls_changed = int(temperatures.isna().sum())

        # Sem valores válidos, não é possível calcular média ou mediana.
        if valid_temperatures.empty:
            treatment_by_component[component] = {
                'method': 'not applied - no valid values',
                'replacement_temperature': None,
                'nulls_changed': 0,
            }
            continue

        # Calcula os quartis e o IQR para identificar outliers.
        q1 = valid_temperatures.quantile(0.25)
        q3 = valid_temperatures.quantile(0.75)
        iqr = q3 - q1
        lower_limit = q1 - 1.5 * iqr
        upper_limit = q3 + 1.5 * iqr
        has_outliers = ((valid_temperatures < lower_limit) | (valid_temperatures > upper_limit)).any()

        # Com outliers, a mediana é mais robusta; sem eles, usa-se a média.
        if has_outliers:
            fill_value = valid_temperatures.median()
            method = 'median (with outliers)'
        else:
            fill_value = valid_temperatures.mean()
            method = 'mean (without outliers)'

        # Preenche somente as temperaturas nulas da coluna atual.
        wide_df[component] = temperatures.fillna(fill_value)
        treatment_by_component[component] = {
            'method': method,
            'replacement_temperature': fill_value,
            'nulls_changed': nulls_changed,
        }

    return treatment_by_component

# Barreira: um termopar medido em menos de COBERTURA_MINIMA_ALVO dos ensaios só
# "sobreviveria" porque a imputação por constante o preencheria, e os modelos o
# tratariam como medição real. Antes de imputar, exige-se uma decisão explícita
# (incluí-lo em EXCLUDED_TC_BY_AREA, ou baixar o corte conscientemente).
termopares_esparsos = []
for area, wide_df in key_area_wide_dataframes.items():
    for component in key_area_dataframes[area]['tc_name'].dropna().unique():
        cobertura = pd.to_numeric(wide_df[component], errors='coerce').notna().mean()
        if cobertura < COBERTURA_MINIMA_ALVO:
            termopares_esparsos.append((area, component, cobertura))
if termopares_esparsos:
    detalhe = '; '.join(f'{a}: {c} ({cob:.1%})' for a, c, cob in termopares_esparsos)
    raise ValueError(
        f'Termopares com cobertura < {COBERTURA_MINIMA_ALVO:.0%} seriam preenchidos quase '
        f'inteiramente por imputação: {detalhe}. Decida sobre cada um em EXCLUDED_TC_BY_AREA.'
    )

# Aplica o tratamento às sete planilhas, usando apenas as colunas de componentes.
null_treatment_summary = {}
for area, wide_df in key_area_wide_dataframes.items():
    component_columns = key_area_dataframes[area]['tc_name'].dropna().unique()
    null_treatment_summary[area] = fill_component_nulls(wide_df, component_columns)

    # Salva a planilha da área já com as temperaturas nulas tratadas.
    filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
    wide_df.to_csv(filename, index=False)
    print(f'"{filename.name}" updated after null-temperature treatment.')

"mainback_data.csv" updated after null-temperature treatment.


"coil_temps_data.csv" updated after null-temperature treatment.
"main_vent_sys_data.csv" updated after null-temperature treatment.
"ipc_cooling_data.csv" updated after null-temperature treatment.


"ipc_comp_data.csv" updated after null-temperature treatment.
"oven_hmi_data.csv" updated after null-temperature treatment.
"ct_knobs_data.csv" updated after null-temperature treatment.


In [8]:
# Reúne apenas os componentes que tiveram valores nulos substituídos.
replacement_records = []
for area, components in null_treatment_summary.items():
    for component, details in components.items():
        if details['nulls_changed'] > 0:
            replacement_records.append({
                'Seção do fogão': area,
                'Componente': component,
                'Temperatura utilizada': details['replacement_temperature'],
                'Método': details['method'],
                'Casas nulas alteradas': details['nulls_changed'],
            })

replacement_report_df = pd.DataFrame(
    replacement_records,
    columns=[
        'Seção do fogão',
        'Componente',
        'Temperatura utilizada',
        'Método',
        'Casas nulas alteradas',
    ],
)

if replacement_report_df.empty:
    print('Nenhum valor nulo foi substituído.')
else:
    display(replacement_report_df)

,Seção do fogão,Componente,Temperatura utilizada,Método,Casas nulas alteradas
0,Mainback,Door Lock Microswitch Body,94.030000,median (with outliers),1
1,Mainback,Hot Surface Indicator Body,42.900000,median (with outliers),1
2,Mainback,Power On Indicator Body,43.050000,median (with outliers),1
3,Coil Temps,Aluminium Plate FR,94.600000,median (with outliers),7
4,Coil Temps,Aluminium Plate FL,122.250000,median (with outliers),7
5,Coil Temps,Aluminium Plate RR,102.620000,median (with outliers),7
6,Coil Temps,Aluminium Plate RL,121.100000,median (with outliers),7
7,IPC Cooling,IPC Housing Surface,71.420000,median (with outliers),2
8,IPC Comp,IPC Blower Housing,63.910000,median (with outliers),5
9,IPC Comp,DCBUS Cap 1,70.405000,median (with outliers),2


## Converter categorias 'Não Informado' pela moda

In [9]:
# Substitui "Não informado" por "No losses" somente quando losses é a única coluna com essa categoria.
TEST_TYPE_COLUMNS = [
    'blower_direction',
    'housing',
    'board_position',
    'run_case',
    'losses',
    'deflector_hole',
    'material_coil_support',
]
MISSING_CATEGORY = 'Não informado'

def is_missing_category(series: pd.Series) -> pd.Series:
    """Identifica 'Não informado' mesmo com espaços extras."""
    return series.astype('string').str.strip().str.casefold().eq(MISSING_CATEGORY.casefold())

columns_with_missing_category = sorted({
    column
    for wide_df in key_area_wide_dataframes.values()
    for column in TEST_TYPE_COLUMNS
    if column in wide_df.columns and is_missing_category(wide_df[column]).any()
})

category_replacement_records = []

if columns_with_missing_category == ['losses']:
    replacement_value = 'No losses'

    for area, wide_df in key_area_wide_dataframes.items():
        if 'losses' not in wide_df.columns:
            continue

        missing_mask = is_missing_category(wide_df['losses'])
        if not missing_mask.any():
            continue

        wide_df.loc[missing_mask, 'losses'] = replacement_value

        # run_id não está nas planilhas finais; recupera os runs afetados no dataset original.
        affected_runs = (
            df.loc[
                df['key_area'].eq(area) & is_missing_category(df['losses']),
                'run_id',
            ]
            .dropna()
            .drop_duplicates()
            .tolist()
        )

        for run_id in affected_runs:
            category_replacement_records.append({
                'Planilha': area,
                'Run alterado': run_id,
                'Tipo de teste': 'losses',
                'Valor inserido': replacement_value,
            })

        # Persiste apenas as planilhas que receberam a correção.
        filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
        wide_df.to_csv(filename, index=False)
else:
    print(
        'Nenhuma alteração realizada: as colunas com Não informado são ' 
        f"{columns_with_missing_category}; esperada apenas ['losses']."
    )

category_replacement_report_df = pd.DataFrame(
    category_replacement_records,
    columns=['Planilha', 'Run alterado', 'Tipo de teste', 'Valor inserido'],
)

if category_replacement_report_df.empty:
    print("Nenhum valor 'Não informado' foi encontrado nos tipos de teste.")
else:
    display(category_replacement_report_df.sort_values(
        ['Planilha', 'Run alterado', 'Tipo de teste']
    ).reset_index(drop=True))


,Planilha,Run alterado,Tipo de teste,Valor inserido
0,CT Knobs,Run 19,losses,No losses
1,CT Knobs,Run 20,losses,No losses
2,CT Knobs,Run 21,losses,No losses
3,CT Knobs,Run 22 (2),losses,No losses
4,CT Knobs,Run 23,losses,No losses
5,Coil Temps,Run 19,losses,No losses
6,Coil Temps,Run 20,losses,No losses
7,Coil Temps,Run 21,losses,No losses
8,Coil Temps,Run 22,losses,No losses
9,Coil Temps,Run 22 (2),losses,No losses


## Converter IGBTs 1 a 16 por apenas 1, utilizando a média

In [10]:
import re

# Consolida IGBT 1 a IGBT 16 em uma única coluna. A média é calculada
# por linha (isto é, para cada configuração/run) e ignora valores ausentes.
PADRAO_IGBT = re.compile(r'^IGBT (?:[1-9]|1[0-6])$')

for area, wide_df in key_area_wide_dataframes.items():
    colunas_igbt = [coluna for coluna in wide_df.columns if isinstance(coluna, str) and PADRAO_IGBT.fullmatch(coluna)]

    if not colunas_igbt:
        continue

    # Converte para número para garantir que a média seja calculada corretamente.
    media_igbt = wide_df[colunas_igbt].apply(pd.to_numeric, errors='coerce').mean(axis=1).round(2)

    # Mantém a nova coluna na posição da primeira coluna IGBT original.
    posicao_igbt = min(wide_df.columns.get_loc(coluna) for coluna in colunas_igbt)
    wide_df = wide_df.drop(columns=colunas_igbt)
    wide_df.insert(posicao_igbt, 'IGBT', media_igbt)

    key_area_wide_dataframes[area] = wide_df
    filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
    wide_df.to_csv(filename, index=False)
    if area == 'IPC Comp':
        print(f'IPC Comp: {len(colunas_igbt)} colunas consolidadas em IGBT e salvo em {filename.name}.')


IPC Comp: 16 colunas consolidadas em IGBT e salvo em ipc_comp_data.csv.


## Unificar dados com configurações iguais

In [11]:
# Unifica medições que possuem a mesma configuração de teste.
# Para cada componente, a temperatura final é a média das medições repetidas.
CONFIGURATION_COLUMNS = [
    'blower_direction',
    'housing',
    'board_position',
    'run_case',
    'losses',
    'deflector_hole',
    'material_coil_support',
]

unification_summary = {}

for area, wide_df in key_area_wide_dataframes.items():
    missing_columns = set(CONFIGURATION_COLUMNS) - set(wide_df.columns)
    if missing_columns:
        raise KeyError(f'{area}: colunas de configuração ausentes: {sorted(missing_columns)}')

    temperature_columns = [
        column for column in wide_df.columns
        if column not in CONFIGURATION_COLUMNS
    ]

    # Garante o cálculo da média mesmo se alguma temperatura estiver como texto.
    temperatures = wide_df[temperature_columns].apply(pd.to_numeric, errors='coerce')
    grouped_df = (
        pd.concat([wide_df[CONFIGURATION_COLUMNS], temperatures], axis=1)
        .groupby(CONFIGURATION_COLUMNS, dropna=False, as_index=False)[temperature_columns]
        .mean()
    )

    unification_summary[area] = {
        'linhas_antes': len(wide_df),
        'linhas_depois': len(grouped_df),
        'configurações_unificadas': len(wide_df) - len(grouped_df),
    }
    key_area_wide_dataframes[area] = grouped_df

    filename = OUTPUT_DIR / f"{area.lower().replace(' ', '_')}_data.csv"
    grouped_df.to_csv(filename, index=False)
    print(
        f'{area}: {len(wide_df)} → {len(grouped_df)} linhas '
        f'({len(wide_df) - len(grouped_df)} configurações repetidas unificadas).'
    )

# Atualiza os DataFrames individuais para refletir os dados unificados.
mainback_wide_df = key_area_wide_dataframes['Mainback']
coil_temps_wide_df = key_area_wide_dataframes['Coil Temps']
main_vent_sys_wide_df = key_area_wide_dataframes['Main Vent Sys']
ipc_cooling_wide_df = key_area_wide_dataframes['IPC Cooling']
ipc_comp_wide_df = key_area_wide_dataframes['IPC Comp']
oven_hmi_wide_df = key_area_wide_dataframes['Oven HMI']
ct_knobs_wide_df = key_area_wide_dataframes['CT Knobs']

display(pd.DataFrame(unification_summary).T)


Mainback: 56 → 32 linhas (24 configurações repetidas unificadas).
Coil Temps: 56 → 32 linhas (24 configurações repetidas unificadas).
Main Vent Sys: 56 → 32 linhas (24 configurações repetidas unificadas).
IPC Cooling: 55 → 32 linhas (23 configurações repetidas unificadas).


IPC Comp: 56 → 32 linhas (24 configurações repetidas unificadas).
Oven HMI: 55 → 32 linhas (23 configurações repetidas unificadas).


CT Knobs: 73 → 32 linhas (41 configurações repetidas unificadas).


,linhas_antes,linhas_depois,configurações_unificadas
Mainback,56,32,24
Coil Temps,56,32,24
Main Vent Sys,56,32,24
IPC Cooling,55,32,23
IPC Comp,56,32,24
Oven HMI,55,32,23
CT Knobs,73,32,41


## Verificação: todas as configurações chegam às planilhas finais

Cada variável independente (as sete colunas de configuração) precisa ter, em cada planilha, todas as categorias da base já padronizada — por exemplo, todas as configurações de `losses`. Uma categoria que não está na planilha é uma configuração que o modelo daquela área nunca verá. A última coluna mostra o que seria perdido se o notebook de treino descartasse as linhas com alvo nulo.

In [12]:
# Base de referência: o dataset longo já padronizado, com "Não informado" tratado
# do mesmo modo que nas planilhas.
base_padronizada = df.copy()
if columns_with_missing_category == ['losses']:
    base_padronizada.loc[is_missing_category(base_padronizada['losses']), 'losses'] = 'No losses'

categorias_base = {
    variavel: set(base_padronizada[variavel].dropna().astype(str))
    for variavel in CONFIGURATION_COLUMNS
}

registros_cobertura = []
for area, wide_df in key_area_wide_dataframes.items():
    # Alvos = termopares reais; a coluna sem nome (tc_name em branco na origem) não é alvo.
    alvos = [
        c for c in wide_df.columns
        if c not in CONFIGURATION_COLUMNS and isinstance(c, str) and c.strip()
        and not wide_df[c].isna().all()
    ]
    sem_alvo_nulo = wide_df.dropna(subset=alvos)
    for variavel in CONFIGURATION_COLUMNS:
        na_planilha = set(wide_df[variavel].dropna().astype(str))
        apos_dropna = set(sem_alvo_nulo[variavel].dropna().astype(str))
        registros_cobertura.append({
            'Planilha': area,
            'Variável': variavel,
            'Categorias na base': len(categorias_base[variavel]),
            'Categorias na planilha': len(na_planilha),
            'Ausentes na planilha': sorted(categorias_base[variavel] - na_planilha),
            'Fora da base': sorted(na_planilha - categorias_base[variavel]),
            'Perdidas se descartar alvo nulo': sorted(na_planilha - apos_dropna),
            'Linhas com alvo nulo': len(wide_df) - len(sem_alvo_nulo),
        })

cobertura_df = pd.DataFrame(registros_cobertura)

# Categorias da base por variável (a lista completa de configurações de cada uma).
for variavel in CONFIGURATION_COLUMNS:
    print(f"{variavel} ({len(categorias_base[variavel])}): {sorted(categorias_base[variavel])}")

# Nenhum termopar excluído pode reaparecer em nenhuma planilha.
for area, wide_df in key_area_wide_dataframes.items():
    reaparecidos = [
        tc for tcs in EXCLUDED_TC_BY_AREA.values() for tc in tcs if tc in wide_df.columns
    ]
    assert not reaparecidos, f'{area}: termopares excluídos presentes: {reaparecidos}'

# Nenhuma grafia divergente pode sobreviver na base final.
for variavel, correcoes in GRAFIAS_DIVERGENTES.items():
    sobreviventes = set(correcoes) & categorias_base[variavel]
    assert not sobreviventes, f'{variavel}: grafias divergentes ainda presentes: {sobreviventes}'

problemas = cobertura_df[
    cobertura_df['Ausentes na planilha'].map(len).gt(0)
    | cobertura_df['Fora da base'].map(len).gt(0)
    | cobertura_df['Perdidas se descartar alvo nulo'].map(len).gt(0)
]
print()
if problemas.empty:
    print('OK - todas as categorias de todas as variáveis estão em todas as planilhas, '
          'sem nenhuma perda por alvo nulo.')
else:
    print('ATENÇÃO - planilhas com configurações ausentes ou em risco:')
    display(problemas.reset_index(drop=True))


blower_direction (2): ['To back', 'To front']
housing (2): ['Closed', 'Open']
board_position (2): ['Center', 'Right']
run_case (4): ['Broil C1', 'Broil C2', 'Broil Qualitative', 'Pyro']
losses (13): ['Broil C2 Simone - 240/145/180/180', 'Broil C2 Simone - Front Order', 'Broil Miro - Front Order', 'C1 Simone - 240/145/180/180', 'Miro', 'Miro - Front Order', 'Miro Qualitative  - Back Order', 'Miro Qualitative  - Front Order', 'No losses', 'Qual Simone - Front Order', 'Simone', 'Simone Losses', 'Simone Modified - Back Order']
deflector_hole (2): ['Closed', 'Open']
material_coil_support (2): ['Aluminium ', 'Plastic ']

OK - todas as categorias de todas as variáveis estão em todas as planilhas, sem nenhuma perda por alvo nulo.
